In [6]:
%uv pip install datasets

Using Python 3.12.6 environment at: /usr/local
Resolved 36 packages in 247ms
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
dill       ------------------------------     0 B/117.21 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 14.84 KiB/117.21 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 14.84 KiB/117.21 KiB
xxhash     ------------------------------ 14.87 KiB/189.39 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 14.84 KiB/117.21 KiB
xxhash     ------------------------------ 14.87 KiB/189.39 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 14.84 KiB/117.21 KiB
xxhash     ------------------------------ 14.87 KiB/189.39 KiB
datasets   ------------------------------     0 B/542.07 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 14.84 KiB/117.21 KiB
xxhash     ------------------------------ 14.87 KiB/189.3

In [1]:
import os
import random
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import matplotlib.pyplot as plt
import pandas as pd
import json
from tqdm import tqdm
from huggingface_hub import login
token=os.environ["HF_TOKEN"]
# login(token=token)
import seaborn as sns
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)



In [24]:
happy_activations = torch.load("happy_activations.pt")
sad_activations = torch.load("sad_activations.pt")

print(happy_activations.shape)
print(sad_activations.shape)

torch.Size([100, 25, 896])
torch.Size([100, 25, 896])


## Calculating the steering vector

In [25]:
happy_acts_23=happy_activations[:,23,:]
sad_acts_23=sad_activations[:,23,:]

happy_mean=happy_acts_23.mean(dim=0)
sad_mean=sad_acts_23.mean(dim=0)

steering_vector=happy_mean-sad_mean

print(steering_vector.shape)

torch.Size([896])


## Neutral prompts for identifying the effect of steering

In [5]:
neutral_prompts = [
    "You open a drawer and find an object you do not remember putting there.",
    "You notice a book left behind on a park bench.",
    "You find an old key while cleaning your room.",
    "You hear music coming from another room but cannot tell where it is from.",
    "You wake up before your alarm and lie in bed for a few minutes.",
    "You find a notebook with the first page torn out.",
    "You take a different route home than usual.",
    "You notice a light still on in a building late at night.",
    "You look out the window and see people walking by.",
    "You walk past a house where someone is playing the piano.",
    "You open your email and see a subject line you were not expecting.",
    "You find a photograph on the ground while walking outside.",
    "You enter a quiet shop you have never been inside before.",
    "You hear your name called in a crowded place.",
    "You find a calendar with an unfamiliar date circled.",
    "You discover a small path branching off from the main road.",
    "You receive a voicemail from an unknown number.",
    "You notice a light flickering in the stairwell.",
    "You see a person standing at a crossroads looking at a piece of paper.",
    "You notice a clock in the room has stopped.",
    "You hear a phone ringing in an empty room.",
    "You arrive at a meeting point and realize you are the first one there.",
    "You notice someone has rearranged the items on your desk.",
    "You find a book with every chapter title highlighted except the last one.",
    "You receive a letter addressed to someone who does not live at your address.",
    "You notice a chair has been moved since you last sat down.",
    "You see a light go on in a building just as you happen to look up.",
    "You walk into a room and cannot remember why you came in.",
    "You find an envelope sealed but with nothing written on the outside.",
    "You notice a car parked in the same spot for several days in a row."
]

In [6]:
len(neutral_prompts)

30

In [19]:
device= torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [20]:
model_id="Qwen/Qwen2.5-0.5B-Instruct"

model=AutoModelForCausalLM.from_pretrained(model_id,trust_remote_code=True).to(device)
tokenizer=AutoTokenizer.from_pretrained(model_id,trust_remote_code=True, device=model.device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
generated_prompts=[f"Write a short first-person diary entry about this situation: {x}" for x in neutral_prompts]

# for i in range(len(neutral_prompts)):
#   prompts_responses[i]={}
#   prompts_responses[i]['prompt']=f"Write a short first-person diary entry about this situation: {neutral_prompts[i]}"
#   prompts_responses[i]['baseline']=""
#   prompts_responses[i]['steering']=""
#   prompts_responses[i]['suppressing']=""

## Additive Steering

In [ ]:
def make_steering_hook(steering_vector, alpha, only_last_token=True):
    def hook(module, input, output):
        hidden_states = output

        steering = steering_vector.to(
            device=hidden_states.device,
            dtype=hidden_states.dtype
        )

        steering = steering.flatten()

        steered_hidden_states = hidden_states.clone()

        if only_last_token:
            steered_hidden_states[:, -1, :] += alpha * steering
        else:
            steered_hidden_states += alpha * steering.view(1, 1, -1)

        return steered_hidden_states

    return hook

In [27]:
def generate_with_steering(
    model,
    tokenizer,
    prompts,
    steering_vector=None,
    layer=23,
    alpha=0.0,
    max_length=128,
    max_new_tokens=256,
    temperature=0.7
):
    if isinstance(prompts, str):
        prompts = [prompts]

    tokenizer.padding_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    texts = []

    for prompt in prompts:
        messages = [
            {"role": "user", "content": prompt}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        texts.append(text)

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    ).to(model.device)

    hook_handle = None

    if steering_vector is not None and alpha != 0.0:
        target_layer = model.model.layers[layer]

        hook_handle = target_layer.register_forward_hook(
            make_steering_hook(steering_vector, alpha, only_last_token=False)
        )

    try:
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.pad_token_id
            )

    finally:
        if hook_handle is not None:
            hook_handle.remove()

    responses = []

    input_len = inputs["input_ids"].shape[1]

    for i in range(len(prompts)):
        generated_ids = output_ids[i, input_len:]

        response = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        responses.append(response)

    return responses

## Sweep across layers and alpha values

In [20]:
layers = list(range(24))
alphas = [-3, -2, -1, -0.5, 0.5, 1, 2, 3]

generation_rows = []

for layer in tqdm(layers, desc="Layers"):
    for alpha in alphas:
        print(f"Layer: {layer}, Alpha: {alpha}")

        responses = generate_with_steering(
            model=model,
            tokenizer=tokenizer,
            prompts=generated_prompts,          # full batch
            steering_vector=steering_vector,
            layer=layer,
            alpha=alpha,
            max_new_tokens=256,
            # temperature=0.7,
            # do_sample=True
        )

        for prompt_id, (prompt, response) in enumerate(zip(generated_prompts, responses)):
            generation_rows.append({
                "prompt_id": prompt_id,
                "prompt": prompt,
                "layer": layer,
                "alpha": alpha,
                "response": response
            })

baseline_responses = generate_with_steering(
    model=model,
    tokenizer=tokenizer,
    prompts=generated_prompts,
    steering_vector=None,
    layer=0,
    alpha=0.0,
    max_new_tokens=256,
    # temperature=0.7,
    # do_sample=True
)

for prompt_id, (prompt, response) in enumerate(zip(generated_prompts, baseline_responses)):
    generation_rows.append({
        "prompt_id": prompt_id,
        "prompt": prompt,
        "layer": None,
        "alpha": 0.0,
        "response": response
    })
generations_df = pd.DataFrame(generation_rows)
generations_df.to_csv("layer_alpha_steering_generations.csv", index=False)

Layers:   0%|                                                                           | 0/24 [00:00<?, ?it/s]

Layer: 0, Alpha: -3
Layer: 0, Alpha: -2
Layer: 0, Alpha: -1
Layer: 0, Alpha: -0.5
Layer: 0, Alpha: 0.5
Layer: 0, Alpha: 1
Layer: 0, Alpha: 2
Layer: 0, Alpha: 3


Layers:   4%|██▊                                                                | 1/24 [01:35<36:40, 95.67s/it]

Layer: 1, Alpha: -3
Layer: 1, Alpha: -2
Layer: 1, Alpha: -1
Layer: 1, Alpha: -0.5
Layer: 1, Alpha: 0.5
Layer: 1, Alpha: 1
Layer: 1, Alpha: 2
Layer: 1, Alpha: 3


Layers:   8%|█████▌                                                             | 2/24 [03:06<33:55, 92.54s/it]

...

Layers:  92%|████████████████████████████████████████████████████████████▌     | 22/24 [35:27<03:07, 93.72s/it]

Layer: 22, Alpha: -3
Layer: 22, Alpha: -2
Layer: 22, Alpha: -1
Layer: 22, Alpha: -0.5
Layer: 22, Alpha: 0.5
Layer: 22, Alpha: 1
Layer: 22, Alpha: 2
Layer: 22, Alpha: 3


Layers:  96%|███████████████████████████████████████████████████████████████▎  | 23/24 [37:00<01:33, 93.45s/it]

Layer: 23, Alpha: -3
Layer: 23, Alpha: -2
Layer: 23, Alpha: -1
Layer: 23, Alpha: -0.5
Layer: 23, Alpha: 0.5
Layer: 23, Alpha: 1
Layer: 23, Alpha: 2
Layer: 23, Alpha: 3


Layers: 100%|██████████████████████████████████████████████████████████████████| 24/24 [38:33<00:00, 96.41s/it]


## Steering using Directional Ablation

In [ ]:
def make_directional_ablation_hook(steering_vector, only_last_token=False, scale=1.0):
    def hook(module, input, output):
        hidden_states = output

        steering = steering_vector.to(
            device=hidden_states.device,
            dtype=hidden_states.dtype
        ).flatten()

        steering = steering / (steering.norm() + 1e-8)

        steered_hidden_states = hidden_states.clone()

        if only_last_token:
            h = steered_hidden_states[:, -1, :]
            coeff = torch.sum(h * steering.view(1, -1), dim=-1, keepdim=True)

            projection = coeff * steering.view(1, -1)

            steered_hidden_states[:, -1, :] = h - scale * projection

        else:
            h = steered_hidden_states

            coeff = torch.sum(h * steering.view(1, 1, -1), dim=-1, keepdim=True)

            projection = coeff * steering.view(1, 1, -1)

            steered_hidden_states = h - scale * projection

        return steered_hidden_states

    return hook

In [11]:
with open("happiness.json", "r") as f:
    happy_data = json.load(f)

happy_prompts=happy_data[101:116]
# neutral_prompts_small=neutral_prompts[0:15]
ablation_prompts=happy_prompts+neutral_prompts
ablation_prompts = [f"Write a short first-person diary entry about this situation: {x}" for x in ablation_prompts]

In [12]:
def generate_with_ablation(
    model,
    tokenizer,
    prompts,
    steering_vector=None,
    max_new_tokens=256,
):
    if isinstance(prompts, str):
        prompts = [prompts]

    tokenizer.padding_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    texts = []

    for prompt in prompts:
        messages = [
            {"role": "user", "content": prompt}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        texts.append(text)

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    hook_handles=[]

    if steering_vector is not None:
        for l in model.model.layers:
            hook_handle = l.register_forward_hook(
                make_directional_ablation_hook(steering_vector,only_last_token=False)
            )
            hook_handles.append(hook_handle)

    try:
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.pad_token_id
            )

    finally:
        if steering_vector is not None:
            for handle in hook_handles:
                handle.remove()

    responses = []

    input_len = inputs["input_ids"].shape[1]

    for i in range(len(prompts)):
        generated_ids = output_ids[i, input_len:]

        response = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        responses.append(response)
        print(response)

    return responses

In [15]:
generation_rows=[]
responses = generate_with_ablation(
            model=model,
            tokenizer=tokenizer,
            prompts=ablation_prompts,    
            max_new_tokens=256,
            steering_vector=steering_vector
            # temperature=0.7,
            # do_sample=True
        )

for prompt_id, (prompt, response) in enumerate(zip(ablation_prompts, responses)):
            generation_rows.append({
                "prompt_id": prompt_id,
                "prompt": prompt,
                "response": response
            })

generations_df = pd.DataFrame(generation_rows)
generations_df.to_csv("directional_ablation_results.csv", index=False)

Dear Diary,

Today was another day filled with new challenges and unexpected opportunities. I found myself in a peculiar predicament that would soon become the catalyst for a journey of self-discovery.

I had always admired a certain type of music that resonated deeply within me, but it seemed so foreign at times. The lyrics were catchy, the melodies memorable, yet they left me feeling both intrigued and slightly overwhelmed. 

One evening, while browsing through my favorite playlists on Spotify, I stumbled upon an artist who seemed to capture the essence of what I longed for in music. Their sound was pure, raw, and incredibly powerful, which immediately caught my attention. It wasn't until then that I realized how much I had missed out on such beauty all those years ago.

The next morning, after a long day of work, I decided to take a break from my usual routine to explore this new musical territory. As I wandered through the quiet streets, I couldn't help but notice the vibrant color

## Testing on Harmful Prompts

In [18]:
data=load_dataset("walledai/HarmBench","standard" ,token=token)

In [23]:
data=data['train']
prompts=data[0:10]['prompt']

In [30]:
layers = list(range(24))
alphas = [-3, -2, -1, -0.5, 0.5, 1, 2, 3]

generation_rows = []

for layer in tqdm(layers, desc="Layers"):
    for alpha in alphas:
        print(f"Layer: {layer}, Alpha: {alpha}")

        responses = generate_with_steering(
            model=model,
            tokenizer=tokenizer,
            prompts=prompts,          # full batch
            steering_vector=steering_vector,
            layer=layer,
            alpha=alpha,
            max_new_tokens=256,
            # temperature=0.7,
            # do_sample=True
        )

        for prompt_id, (prompt, response) in enumerate(zip(prompts, responses)):
            generation_rows.append({
                "prompt_id": prompt_id,
                "prompt": prompt,
                "layer": layer,
                "alpha": alpha,
                "response": response
            })

generations_df = pd.DataFrame(generation_rows)
generations_df.to_csv("harmbench_steering_responses.csv", index=False)

Layers:   0%|                                                                           | 0/24 [00:00<?, ?it/s]

Layer: 0, Alpha: -3
Layer: 0, Alpha: -2
Layer: 0, Alpha: -1
Layer: 0, Alpha: -0.5
Layer: 0, Alpha: 0.5
Layer: 0, Alpha: 1
Layer: 0, Alpha: 2
Layer: 0, Alpha: 3


Layers:   4%|██▊                                                                | 1/24 [01:39<38:11, 99.64s/it]

Layer: 1, Alpha: -3
Layer: 1, Alpha: -2
Layer: 1, Alpha: -1
Layer: 1, Alpha: -0.5
Layer: 1, Alpha: 0.5
Layer: 1, Alpha: 1
Layer: 1, Alpha: 2
Layer: 1, Alpha: 3


Layers:   8%|█████▌                                                             | 2/24 [03:12<35:07, 95.79s/it]

...

Layers:  92%|████████████████████████████████████████████████████████████▌     | 22/24 [33:30<02:53, 86.56s/it]

Layer: 22, Alpha: -3
Layer: 22, Alpha: -2
Layer: 22, Alpha: -1
Layer: 22, Alpha: -0.5
Layer: 22, Alpha: 0.5
Layer: 22, Alpha: 1
Layer: 22, Alpha: 2
Layer: 22, Alpha: 3


Layers:  96%|███████████████████████████████████████████████████████████████▎  | 23/24 [34:59<01:27, 87.15s/it]

Layer: 23, Alpha: -3
Layer: 23, Alpha: -2
Layer: 23, Alpha: -1
Layer: 23, Alpha: -0.5
Layer: 23, Alpha: 0.5
Layer: 23, Alpha: 1
Layer: 23, Alpha: 2
Layer: 23, Alpha: 3


Layers: 100%|██████████████████████████████████████████████████████████████████| 24/24 [36:29<00:00, 91.25s/it]


In [31]:
del model
del tokenizer

In [36]:
%uv pip install judgezoo 
%uv pip install -U torchao

Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 33ms


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Using Python 3.12.6 environment at: /usr/local
Resolved 1 package in 41ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
Prepared 1 package in 228ms
Installed 1 package in 43ms
 + torchao==0.17.0
Note: you may need to restart the kernel to use updated packages.


## Harmfulness Scores

In [18]:
import judgezoo
from judgezoo import Judge
judge = Judge.from_name("strong_reject")

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

In [15]:
harm_responses=pd.read_csv("harmbench_steering_responses.csv")

In [ ]:
def make_conversation(prompt, response):
    return [
        {
            "role": "user",
            "content": str(prompt)
        },
        {
            "role": "assistant",
            "content": str(response)
        }
    ]

In [ ]:
def make_conversations(df, prompt_col="prompt", response_col="response"):
    conversations = []

    for _, row in df.iterrows():
        conv = make_conversation(
            prompt=row[prompt_col],
            response=row[response_col]
        )
        conversations.append(conv)

    return conversations


In [ ]:
def get_harmful_scores(conversations, judge):
    harmful_scores = []
    raw_scores = []

    for conv in tqdm(conversations, desc="Running JudgeZoo"):
        try:
            score = judge([conv])

            raw_scores.append(score)

            # Usually strong_reject returns p_harmful
            harmful_scores.append(score.get("p_harmful", np.nan))

        except Exception as e:
            print("Judge failed on one example:", e)
            raw_scores.append({"error": str(e)})
            harmful_scores.append(np.nan)

    return harmful_scores, raw_scores

In [19]:
conversations = make_conversations(harm_responses)

harmful_scores, raw_scores = get_harmful_scores(
    conversations=conversations,
    judge=judge
)

harm_responses["harmful_score"] = harmful_scores
harm_responses["judgezoo_raw"] = raw_scores

Running JudgeZoo: 100%|████████████████████████████████████████████████████| 1920/1920 [31:07<00:00,  1.03it/s]


In [21]:
harm_responses.to_csv("harm_responses_judged.csv", index=False)